In [1]:
import os

# 建立資料夾
os.makedirs("project_gutenberg", exist_ok=True)

# 建立檔案
'''
 with open("project_gutenberg/project_gutenberg.py", "w") as f:
    pass
 '''
with open("project_gutenberg/README.md", "w") as f:
    pass

In [ ]:
'''
匯入套件
'''
# 操作 browser 的 API
from selenium.webdriver.chrome.service import Service
from selenium import webdriver
from selenium.webdriver.support.ui import Select

# 處理逾時例外的工具
from selenium.common.exceptions import TimeoutException

# 面對動態網頁，等待某個元素出現的工具，通常與 exptected_conditions 搭配
from selenium.webdriver.support.ui import WebDriverWait

# 搭配 WebDriverWait 使用，對元素狀態的一種期待條件，若條件發生，則等待結束，往下一行執行
from selenium.webdriver.support import expected_conditions as EC

# 期待元素出現要透過什麼方式指定，通常與 EC、WebDriverWait 一起使用
from selenium.webdriver.common.by import By

# 強制等待 (執行期間休息一下)
from time import sleep

from bs4 import BeautifulSoup
import time
import os
import re



# 設定參數
MAX_BOOKS = 250
MIN_CHINESE_CHARACTERS = 1000
output_folder = "project_gutenberg"
os.makedirs(output_folder, exist_ok=True)

# 正規表示式：只保留中文字與常用標點符號
chinese_pattern = re.compile(r"[\u4e00-\u9fff\u3000-\u303f\uff01-\uff5e，。、：「」『』？！…—《》〈〉（）〔〕【】]+", re.UNICODE)

# 建立瀏覽器
options = webdriver.ChromeOptions()
# options.add_argument("--headless")  # 無頭模式可取消註解
driver = webdriver.Chrome(service=Service(), options=options)

# 進入 Project Gutenberg 中文書籍分類頁
driver.get("https://www.gutenberg.org/browse/languages/zh")
time.sleep(2)

# Step 1: 把書籍的連結與名稱存起來，避免 stale element 問題
link_infos = []
elements = driver.find_elements(By.CSS_SELECTOR, "li.pgdbetext > a[href]")
for elem in elements:
    title = elem.text.strip().replace("/", "_").replace("\\", "_")
    href = elem.get_attribute("href")
    if title and href:
        link_infos.append((title, href))

print(f"共找到 {len(link_infos)} 本書")

# Step 2: 開始下載每一本書
count = 0
for book_name, href in link_infos:
    try:
        driver.get(href)
        # time.sleep(2)

        # 尋找含 .txt 的純文字連結
        txt_links = driver.find_elements(By.CSS_SELECTOR, 'a[href*=".txt"]')
        if not txt_links:
            continue

        txt_url = txt_links[0].get_attribute("href")
        driver.get(txt_url)
        # time.sleep(1)

        # 取得純文字內容
        soup = BeautifulSoup(driver.page_source, "html.parser")
        text = soup.get_text()
        chinese_text = "".join(chinese_pattern.findall(text))

        # 過濾太少字數的內容
        if len(chinese_text) < MIN_CHINESE_CHARACTERS:
            continue

        # 儲存為 .txt
        save_path = os.path.join(output_folder, f"{book_name}.txt")
        with open(save_path, "w", encoding="utf-8") as f:
            f.write(chinese_text)

        count += 1
        print(f"[{count}] 儲存：{book_name}.txt")

        if count >= MAX_BOOKS:
            break

    except Exception as e:
        print(f"錯誤 {book_name}：{e}")
        continue

driver.quit()
print(f"完成，共儲存 {count} 本書。")

共找到 507 本書
[1] 儲存：豆棚閒話.txt
[2] 儲存：戲中戲.txt
[3] 儲存：比目魚.txt
[4] 儲存：比目魚.txt
[5] 儲存：三字經.txt
[6] 儲存：山水情.txt
[7] 儲存：山海經.txt
[8] 儲存：施公案.txt
[9] 儲存：施公案.txt
[10] 儲存：易經.txt
[11] 儲存：木蘭奇女傳.txt
[12] 儲存：海公案.txt
[13] 儲存：燕丹子.txt
[14] 儲存：狄公案.txt
[15] 儲存：禮記.txt
[16] 儲存：綠牡丹.txt
[17] 儲存：詩經.txt
[18] 儲存：麟兒報.txt
錯誤 Hu Die Mei
Yuan Yang Meng：[Errno 22] Invalid argument: 'project_gutenberg\\Hu Die Mei\nYuan Yang Meng.txt'
錯誤 Qing Lou Meng
Qi Hong Xiao Shi：[Errno 22] Invalid argument: 'project_gutenberg\\Qing Lou Meng\nQi Hong Xiao Shi.txt'
[19] 儲存：天豹圖.txt
[20] 儲存：梁公九諫.txt
[21] 儲存：李娃傳.txt
[22] 儲存：玉樓春.txt
[23] 儲存：漢書.txt
[24] 儲存：引鳳蕭.txt
[25] 儲存：今古奇觀.txt
[26] 儲存：後西游記.txt
[27] 儲存：飛跎全傳.txt
[28] 儲存：佛說四十二章經.txt
[29] 儲存：紅樓夢.txt
[30] 儲存：洛神賦.txt
[31] 儲存：晁氏儒言 一卷.txt
[32] 儲存：水滸後傳.txt
[33] 儲存：幼學瓊林.txt
[34] 儲存：治世餘聞.txt
[35] 儲存：琵琶記.txt
[36] 儲存：雪月梅傳.txt
[37] 儲存：龍川詞.txt
[38] 儲存：三國志.txt
[39] 儲存：隋唐演義.txt
[40] 儲存：論語.txt
[41] 儲存：滬語開路 = Conversational Exercises in the Shanghai Dialect.txt
[42] 儲存：白圭志.txt
[43] 儲存：孟子字義疏證.txt
[44] 儲存：安樂